# Ants and bees image classification

This notebook compares six approaches to classifying photographs of ants and bees. Each model has its own architecture, training progress, learning curves, precision, recall, F1 score, and confusion matrix. A final section compares the six results with the previous notebook run.

**What changed.** Training crops now keep more of each insect. CNN2D and CNN3D have stronger feature blocks and longer schedules. The two recurrent models now read rows of **frozen pretrained ResNet18 features** instead of raw pixel rows. The partial fine-tune run updates only the final ResNet block and classifier. These changes make the recurrent runs hybrid vision and sequence models, so their results should not be read as a comparison of plain RNN or LSTM against a CNN.

**How to interpret the scores.** All reported scores use the same 153-image validation folder. The best epoch was selected using that folder, and these experiments were refined after inspecting earlier validation results. There is no untouched test set; the numbers describe these validation runs and may be optimistic for unseen images. CNN3D uses RGB channels as a short depth axis, not true 3D imagery or video.

Run from a Jupyter kernel installed in your own environment. Set `VISION_DATA_DIR` to an externally acquired folder containing `train/` and `val/`. Original outputs were cleared for privacy and image rights; exact historical numeric results remain in `results/model_results.json`. New runs never overwrite that artifact.


## 1. Imports and run settings

In [ ]:
import os
import sys
from copy import deepcopy
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from vision_comparison import CNN2DImproved, CNN3DImproved, PretrainedRowSequence
from PIL import Image
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = False

# These seeds and schedules correspond to the results shown below.
SEEDS = {"CNN2D": 142, "CNN3D (RGB depth)": 143,
         "RNN (ResNet features)": 142, "LSTM (ResNet features)": 143,
         "ResNet18 feature extractor": 144, "ResNet18 partial fine-tune": 145}
EPOCHS = {"CNN2D": 28, "CNN3D (RGB depth)": 28,
          "RNN (ResNet features)": 14, "LSTM (ResNet features)": 14,
          "ResNet18 feature extractor": 16, "ResNet18 partial fine-tune": 14}
BATCH_SIZE = 16
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
print(f"Libraries imported successfully | PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
display(pd.DataFrame({"epochs": EPOCHS, "seed": SEEDS}).T.T)

## 2. Dataset and preprocessing

The supplied Hymenoptera dataset has 244 training photographs and 153 validation photographs. Augmentation applies **only to training**: a crop retaining at least 65% of image area, a small rotation, horizontal flip, and light color variation. Validation preprocessing is fixed. Pretrained models use 224 × 224 images with ImageNet normalization. The scratch CNNs use 128 × 128 images normalized to the range around zero.

In [ ]:
DATA_DIR = Path(os.environ.get("VISION_DATA_DIR", str(ROOT / "data" / "hymenoptera_data"))).expanduser()
if not (DATA_DIR / "train").is_dir() or not (DATA_DIR / "val").is_dir():
    raise FileNotFoundError("Set VISION_DATA_DIR to an authorized Hymenoptera dataset containing train/ and val/.")
raw_train = datasets.ImageFolder(DATA_DIR / "train")
raw_val = datasets.ImageFolder(DATA_DIR / "val")
class_names = raw_train.classes
if raw_val.classes != class_names:
    raise ValueError("Training and validation class folders differ.")
class_counts = pd.DataFrame({"Train": np.bincount(raw_train.targets, minlength=2),
                             "Validation": np.bincount(raw_val.targets, minlength=2)}, index=class_names)
class_counts.loc["Total"] = class_counts.sum()
print(f"Dataset loaded successfully: {DATA_DIR}")
display(class_counts)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
class_counts.drop(index="Total").plot.bar(ax=axes[0], rot=0, color=["#163F5B", "#C46332"])
axes[0].set(title="Class counts", xlabel="", ylabel="Images")
for ax, label in zip(axes[1:], class_names):
    image_path = next((DATA_DIR / "train" / label).glob("*.jpg"))
    with Image.open(image_path) as sample:
        ax.imshow(sample.convert("RGB"))
    ax.set_title(f"Training example: {label}")
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)

def make_loaders(size: int, batch_size: int = 16):
    mean, std = (MEAN, STD) if size == 224 else ([0.5] * 3, [0.5] * 3)
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.65, 1.0), ratio=(0.85, 1.15)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(12),
        transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(round(size * 256 / 224)),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    datasets_by_phase = {
        "train": datasets.ImageFolder(DATA_DIR / "train", transform=train_tf),
        "val": datasets.ImageFolder(DATA_DIR / "val", transform=val_tf),
    }
    return {
        phase: DataLoader(
            ds, batch_size=batch_size, shuffle=(phase == "train"), num_workers=0,
            pin_memory=(DEVICE.type == "cuda"),
        )
        for phase, ds in datasets_by_phase.items()
    }

resnet_loaders = make_loaders(224, BATCH_SIZE)
scratch_loaders = make_loaders(128, BATCH_SIZE)
print("Data loaders ready | pretrained encoder: 224 × 224 | scratch CNNs: 128 × 128")

## 3. Shared training and evaluation functions

Every section uses the same optimizer family, cosine learning-rate decay, and smoothed cross-entropy loss. The progress bar shows batches completed, phase, running accuracy, and loss. We restore the checkpoint with the highest validation accuracy; if accuracy ties, lower validation loss decides. The validation set also supplies the final metrics, so this remains a descriptive result rather than a held-out test estimate.

In [ ]:
def train_model(model, loaders, name, optimizer, epochs, freeze_mode=None):
    # One progress bar covers both phases across all epochs.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
    best = {"accuracy": -1.0, "loss": float("inf"), "epoch": 0, "weights": None}
    history = []
    started = time.perf_counter()
    total_batches = epochs * sum(len(loader) for loader in loaders.values())
    with tqdm(total=total_batches, desc=name, unit="batch", dynamic_ncols=True, mininterval=0.5) as bar:
        for epoch in range(1, epochs + 1):
            row = {"epoch": epoch}
            for phase in ("train", "val"):
                is_training = phase == "train"
                model.train(is_training)
                if is_training and freeze_mode == "feature":
                    model.eval()       # Keep the frozen backbone and its batch statistics fixed.
                    model.fc.train()
                elif is_training and freeze_mode == "partial":
                    model.eval()       # Frozen early blocks stay in evaluation mode.
                    model.layer4.train()
                    model.fc.train()
                elif is_training and freeze_mode == "sequence":
                    model.backbone.eval()
                total_loss = total_correct = total_seen = 0
                for images, labels in loaders[phase]:
                    images = images.to(DEVICE, non_blocking=True)
                    labels = labels.to(DEVICE, non_blocking=True)
                    if is_training:
                        optimizer.zero_grad(set_to_none=True)
                    with torch.set_grad_enabled(is_training):
                        logits = model(images)
                        loss = criterion(logits, labels)
                        if is_training:
                            loss.backward()
                            optimizer.step()
                    count = labels.size(0)
                    total_seen += count
                    total_loss += loss.item() * count
                    total_correct += (logits.argmax(dim=1) == labels).sum().item()
                    bar.update(1)
                    bar.set_postfix(epoch=f"{epoch}/{epochs}", phase=phase,
                                    loss=f"{total_loss / total_seen:.3f}",
                                    accuracy=f"{total_correct / total_seen:.1%}")
                row[f"{phase}_loss"] = total_loss / total_seen
                row[f"{phase}_accuracy"] = total_correct / total_seen
            scheduler.step()
            history.append(row)
            if (row["val_accuracy"] > best["accuracy"] or
                (row["val_accuracy"] == best["accuracy"] and row["val_loss"] < best["loss"])):
                best.update(accuracy=row["val_accuracy"], loss=row["val_loss"],
                            epoch=epoch, weights=deepcopy(model.state_dict()))
    model.load_state_dict(best["weights"])
    model.eval()
    history = pd.DataFrame(history)
    elapsed = (time.perf_counter() - started) / 60
    print(f"{name}: {elapsed:.1f} min | best validation accuracy {best['accuracy']:.1%} at epoch {best['epoch']}")
    display(history.round(3))
    return model, history, best["epoch"], elapsed

trained_models, histories, evaluations, best_epochs, minutes_by_model = {}, {}, {}, {}, {}

In [ ]:
def evaluate_model(model, loader, name, best_epoch):
    # Evaluate the restored best checkpoint on every validation photograph.
    model.eval()
    truth, predictions = [], []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            truth.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    ids = list(range(len(class_names)))
    macro = precision_recall_fscore_support(truth, predictions, labels=ids, average="macro", zero_division=0)
    weighted = precision_recall_fscore_support(truth, predictions, labels=ids, average="weighted", zero_division=0)
    report = classification_report(truth, predictions, labels=ids, target_names=class_names,
                                   output_dict=True, zero_division=0)
    report.pop("accuracy")
    summary = {
        "Model": name, "Best epoch": int(best_epoch), "Validation samples": len(truth),
        "Accuracy": float(accuracy_score(truth, predictions)),
        "Precision (macro)": float(macro[0]), "Recall (macro)": float(macro[1]), "F1 (macro)": float(macro[2]),
        "Precision (weighted)": float(weighted[0]), "Recall (weighted)": float(weighted[1]),
        "F1 (weighted)": float(weighted[2]),
    }
    matrix = confusion_matrix(truth, predictions, labels=ids)
    return summary, pd.DataFrame(report).T, matrix


def show_model_results(name, history, evaluation):
    # Keep the learning curves, confusion matrix, and class metrics together.
    summary, report, matrix = evaluation
    print(f"{name}: restored checkpoint from epoch {summary['Best epoch']}")
    display(pd.DataFrame([summary]).set_index("Model").round(3))
    print("Per-class precision, recall, F1 score, and sample count:")
    display(report.round(3))

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    train_color, val_color = "#163F5B", "#C46332"
    for ax, metric, label in ((axes[0, 0], "accuracy", "Accuracy"),
                              (axes[0, 1], "loss", "Cross-entropy loss")):
        ax.plot(history.epoch, history[f"train_{metric}"], color=train_color, label="Train", linewidth=2)
        ax.plot(history.epoch, history[f"val_{metric}"], color=val_color, label="Validation", linewidth=2)
        ax.axvline(summary["Best epoch"], color="#73808B", linestyle="--", alpha=0.8)
        ax.set(title=f"Train and validation {metric}", xlabel="Epoch", ylabel=label)
        ticks = np.unique(np.linspace(1, len(history), min(8, len(history)), dtype=int))
        ax.set_xticks(ticks)
        if metric == "accuracy": ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.18)
        ax.legend(frameon=False)
    ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
        ax=axes[1, 0], cmap="Blues", colorbar=False, values_format="d")
    axes[1, 0].set_title("Confusion matrix: actual rows, predicted columns")
    report.loc[class_names, ["precision", "recall", "f1-score"]].plot.bar(
        ax=axes[1, 1], rot=0, ylim=(0, 1.04), color=[train_color, val_color, "#557D61"])
    axes[1, 1].set(title="Per-class validation scores", xlabel="Class", ylabel="Score")
    axes[1, 1].grid(axis="y", alpha=0.18)
    fig.suptitle(f"{name}: training and validation", fontsize=16)
    fig.tight_layout()
    plt.show()

## 4. ResNet18 partial fine-tune

Start from ImageNet weights. Freeze the early visual layers and update only the final residual block and a new two-class head. The final block uses a lower learning rate than the head. This limits how much the small dataset can disturb useful pretrained features.

In [ ]:
seed_all(SEEDS["ResNet18 partial fine-tune"])
resnet_ft = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in resnet_ft.parameters():
    parameter.requires_grad = False
for parameter in resnet_ft.layer4.parameters():
    parameter.requires_grad = True
resnet_ft.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(512, len(class_names)))
resnet_ft = resnet_ft.to(DEVICE)
resnet_ft_optimizer = torch.optim.AdamW([
    {"params": resnet_ft.layer4.parameters(), "lr": 1e-4},
    {"params": resnet_ft.fc.parameters(), "lr": 7e-4},
], weight_decay=1e-4)
print(f"ResNet18 partial fine-tune | trainable parameters: {sum(p.numel() for p in resnet_ft.parameters() if p.requires_grad):,}")

In [ ]:
resnet_ft, resnet_ft_history, resnet_ft_best_epoch, resnet_ft_minutes = train_model(
    resnet_ft, resnet_loaders, "ResNet18 partial fine-tune", resnet_ft_optimizer, EPOCHS["ResNet18 partial fine-tune"], freeze_mode="partial"
)
trained_models["ResNet18 partial fine-tune"] = resnet_ft
histories["ResNet18 partial fine-tune"] = resnet_ft_history
best_epochs["ResNet18 partial fine-tune"] = resnet_ft_best_epoch
minutes_by_model["ResNet18 partial fine-tune"] = resnet_ft_minutes

In [ ]:
resnet_ft_results = evaluate_model(resnet_ft, resnet_loaders["val"], "ResNet18 partial fine-tune", resnet_ft_best_epoch)
evaluations["ResNet18 partial fine-tune"] = resnet_ft_results
show_model_results("ResNet18 partial fine-tune", resnet_ft_history, resnet_ft_results)

## 5. ResNet18 feature extractor

Keep the entire ImageNet backbone fixed, including batch-normalization statistics. Train a small two-layer head with dropout. This tests whether pretrained image features already separate ants and bees well.

In [ ]:
seed_all(SEEDS["ResNet18 feature extractor"])
resnet_fe = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in resnet_fe.parameters():
    parameter.requires_grad = False
resnet_fe.fc = nn.Sequential(nn.Dropout(0.25), nn.Linear(512, 128), nn.ReLU(),
                             nn.Dropout(0.2), nn.Linear(128, len(class_names)))
resnet_fe = resnet_fe.to(DEVICE)
resnet_fe_optimizer = torch.optim.AdamW((p for p in resnet_fe.parameters() if p.requires_grad),
                                       lr=8e-4, weight_decay=1e-4)
print(f"ResNet18 feature extractor | trainable parameters: {sum(p.numel() for p in resnet_fe.parameters() if p.requires_grad):,}")

In [ ]:
resnet_fe, resnet_fe_history, resnet_fe_best_epoch, resnet_fe_minutes = train_model(
    resnet_fe, resnet_loaders, "ResNet18 feature extractor", resnet_fe_optimizer, EPOCHS["ResNet18 feature extractor"], freeze_mode="feature"
)
trained_models["ResNet18 feature extractor"] = resnet_fe
histories["ResNet18 feature extractor"] = resnet_fe_history
best_epochs["ResNet18 feature extractor"] = resnet_fe_best_epoch
minutes_by_model["ResNet18 feature extractor"] = resnet_fe_minutes

In [ ]:
resnet_fe_results = evaluate_model(resnet_fe, resnet_loaders["val"], "ResNet18 feature extractor", resnet_fe_best_epoch)
evaluations["ResNet18 feature extractor"] = resnet_fe_results
show_model_results("ResNet18 feature extractor", resnet_fe_history, resnet_fe_results)

## 6. CNN2D trained from scratch

A residual 2D CNN learns spatial filters from the training images without ImageNet weights. Its skip connections help gradients reach the deeper blocks. This remains the cleanest scratch convolutional baseline in the notebook.

In [ ]:
seed_all(SEEDS["CNN2D"])
cnn2d = CNN2DImproved().to(DEVICE)
cnn2d_optimizer = torch.optim.AdamW(cnn2d.parameters(), lr=1e-3, weight_decay=1e-4)
print(f"CNN2D | trainable parameters: {sum(p.numel() for p in cnn2d.parameters() if p.requires_grad):,}")

In [ ]:
cnn2d, cnn2d_history, cnn2d_best_epoch, cnn2d_minutes = train_model(
    cnn2d, scratch_loaders, "CNN2D", cnn2d_optimizer, EPOCHS["CNN2D"]
)
trained_models["CNN2D"] = cnn2d
histories["CNN2D"] = cnn2d_history
best_epochs["CNN2D"] = cnn2d_best_epoch
minutes_by_model["CNN2D"] = cnn2d_minutes

In [ ]:
cnn2d_results = evaluate_model(cnn2d, scratch_loaders["val"], "CNN2D", cnn2d_best_epoch)
evaluations["CNN2D"] = cnn2d_results
show_model_results("CNN2D", cnn2d_history, cnn2d_results)

## 7. CNN3D using RGB as depth

A deeper 3D CNN treats the three RGB channels as a short depth axis while pooling spatial dimensions. It remains an educational use of 3D convolution on still photographs; RGB depth is not physical scene depth.

In [ ]:
seed_all(SEEDS["CNN3D (RGB depth)"])
cnn3d = CNN3DImproved().to(DEVICE)
cnn3d_optimizer = torch.optim.AdamW(cnn3d.parameters(), lr=1e-3, weight_decay=1e-4)
print(f"CNN3D | trainable parameters: {sum(p.numel() for p in cnn3d.parameters() if p.requires_grad):,}")

In [ ]:
cnn3d, cnn3d_history, cnn3d_best_epoch, cnn3d_minutes = train_model(
    cnn3d, scratch_loaders, "CNN3D (RGB depth)", cnn3d_optimizer, EPOCHS["CNN3D (RGB depth)"]
)
trained_models["CNN3D (RGB depth)"] = cnn3d
histories["CNN3D (RGB depth)"] = cnn3d_history
best_epochs["CNN3D (RGB depth)"] = cnn3d_best_epoch
minutes_by_model["CNN3D (RGB depth)"] = cnn3d_minutes

In [ ]:
cnn3d_results = evaluate_model(cnn3d, scratch_loaders["val"], "CNN3D (RGB depth)", cnn3d_best_epoch)
evaluations["CNN3D (RGB depth)"] = cnn3d_results
show_model_results("CNN3D (RGB depth)", cnn3d_history, cnn3d_results)

## 8. Shared pretrained row-sequence encoder

The RNN and LSTM now receive seven rows of **frozen ResNet18 spatial features**. Averaging the width dimension leaves a seven-step sequence of 512-feature vectors. The recurrent layer and classifier are trainable. This substantially changes the experiment from raw-pixel sequence modeling: the pretrained CNN does much of the visual recognition before the RNN or LSTM sees the image.

In [ ]:
# Reusable, kind-validated frozen-backbone implementation lives in src/vision_comparison.py.


## 9. RNN on pretrained visual features

A bidirectional vanilla RNN reads the seven feature rows. It combines sequence information from both vertical directions before the final classifier.

In [ ]:
seed_all(SEEDS["RNN (ResNet features)"])
rnn = PretrainedRowSequence("RNN").to(DEVICE)
rnn_optimizer = torch.optim.AdamW((p for p in rnn.parameters() if p.requires_grad),
                                  lr=1e-3, weight_decay=1e-4)
print(f"RNN hybrid | trainable parameters: {sum(p.numel() for p in rnn.parameters() if p.requires_grad):,}")

In [ ]:
rnn, rnn_history, rnn_best_epoch, rnn_minutes = train_model(
    rnn, resnet_loaders, "RNN (ResNet features)", rnn_optimizer, EPOCHS["RNN (ResNet features)"], freeze_mode="sequence"
)
trained_models["RNN (ResNet features)"] = rnn
histories["RNN (ResNet features)"] = rnn_history
best_epochs["RNN (ResNet features)"] = rnn_best_epoch
minutes_by_model["RNN (ResNet features)"] = rnn_minutes

In [ ]:
rnn_results = evaluate_model(rnn, resnet_loaders["val"], "RNN (ResNet features)", rnn_best_epoch)
evaluations["RNN (ResNet features)"] = rnn_results
show_model_results("RNN (ResNet features)", rnn_history, rnn_results)

## 10. LSTM on pretrained visual features

The LSTM receives the same feature-sequence structure and uses gated memory. The RNN and LSTM runs use different seeds and selected epochs; their difference cannot be attributed solely to recurrent cell choice.


In [ ]:
seed_all(SEEDS["LSTM (ResNet features)"])
lstm = PretrainedRowSequence("LSTM").to(DEVICE)
lstm_optimizer = torch.optim.AdamW((p for p in lstm.parameters() if p.requires_grad),
                                   lr=1e-3, weight_decay=1e-4)
print(f"LSTM hybrid | trainable parameters: {sum(p.numel() for p in lstm.parameters() if p.requires_grad):,}")

In [ ]:
lstm, lstm_history, lstm_best_epoch, lstm_minutes = train_model(
    lstm, resnet_loaders, "LSTM (ResNet features)", lstm_optimizer, EPOCHS["LSTM (ResNet features)"], freeze_mode="sequence"
)
trained_models["LSTM (ResNet features)"] = lstm
histories["LSTM (ResNet features)"] = lstm_history
best_epochs["LSTM (ResNet features)"] = lstm_best_epoch
minutes_by_model["LSTM (ResNet features)"] = lstm_minutes

In [ ]:
lstm_results = evaluate_model(lstm, resnet_loaders["val"], "LSTM (ResNet features)", lstm_best_epoch)
evaluations["LSTM (ResNet features)"] = lstm_results
show_model_results("LSTM (ResNet features)", lstm_history, lstm_results)

## 11. Validation comparison and change from the previous run

The first table ranks the six updated runs. The second compares each architecture family with the earlier notebook by the number of correctly classified validation images. Because training schedules, augmentations, architectures, and some seeds changed, the difference is a combined result of those choices rather than the effect of one isolated change. The recurrent models also gained a pretrained visual encoder.

In [ ]:
MODEL_ORDER = ["ResNet18 partial fine-tune", "ResNet18 feature extractor", "CNN2D",
               "CNN3D (RGB depth)", "RNN (ResNet features)", "LSTM (ResNet features)"]
results_df = pd.DataFrame([evaluations[name][0] for name in MODEL_ORDER]).set_index("Model")
print("Updated validation results, ranked by accuracy:")
display(results_df.sort_values("Accuracy", ascending=False).round(3))

# The previous validation scores are retained here for a transparent before/after view.
baseline_accuracy = {
    "ResNet18 partial fine-tune": 0.934640522875817,
    "ResNet18 feature extractor": 0.9607843137254902,
    "CNN2D": 0.7189542483660131,
    "CNN3D (RGB depth)": 0.7320261437908496,
    "RNN (ResNet features)": 0.5424836601307189,
    "LSTM (ResNet features)": 0.5882352941176471,
}
change_df = pd.DataFrame({
    "Previous accuracy": pd.Series(baseline_accuracy),
    "Updated accuracy": results_df["Accuracy"],
})
change_df["More correct images"] = ((change_df["Updated accuracy"] -
                                      change_df["Previous accuracy"]) * len(raw_val)).round().astype(int)
display(change_df.sort_values("Updated accuracy", ascending=False).round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
results_df["Accuracy"].sort_values().plot.barh(ax=axes[0], color="#163F5B", xlim=(0, 1))
axes[0].set(title="Updated validation accuracy", xlabel="Accuracy", ylabel="")
change_df.sort_values("Updated accuracy")[["Previous accuracy", "Updated accuracy"]].plot.barh(
    ax=axes[1], color=["#B6C2C8", "#C46332"], xlim=(0, 1))
axes[1].set(title="Previous and updated validation accuracy", xlabel="Accuracy", ylabel="")
for ax in axes:
    ax.grid(axis="x", alpha=0.18)
fig.tight_layout()
plt.show()

In [ ]:
# Export the exact results for the presentation and script.
RUN_DIR = ROOT / "results" / "runs" / time.strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_PATH = RUN_DIR / "model_results.json"
model_notes = {
    "ResNet18 partial fine-tune": "ImageNet ResNet18; layer4 and new head train; 224 px",
    "ResNet18 feature extractor": "Frozen ImageNet ResNet18; MLP head trains; 224 px",
    "CNN2D": "Residual 2D CNN trained from scratch; 128 px",
    "CNN3D (RGB depth)": "3D CNN trained from scratch; RGB is depth axis; 128 px",
    "RNN (ResNet features)": "Bidirectional RNN on seven rows of frozen ResNet18 features; 224 px",
    "LSTM (ResNet features)": "Bidirectional LSTM on seven rows of frozen ResNet18 features; 224 px",
}
payload = {
    "title": "Ants and bees: six improved image classification models",
    "dataset": {
        "path": "external dataset (not redistributed)", "classes": class_names, "train_count": len(raw_train),
        "validation_count": len(raw_val),
        "class_counts": json.loads(class_counts.to_json(orient="index")),
        "note": "Validation selected each best epoch and supplied metrics; no independent test set.",
    },
    "settings": {"batch_size": BATCH_SIZE, "device": str(DEVICE),
                 "epochs_by_model": EPOCHS, "seeds_by_model": SEEDS,
                 "scratch_image_size": 128, "pretrained_image_size": 224,
                 "augmentation": "crop scale 0.65-1.0, horizontal flip, rotation 12 degrees, light color jitter"},
    "models": {
        name: {"history": json.loads(histories[name].to_json(orient="records")),
               "summary": evaluations[name][0],
               "per_class": json.loads(evaluations[name][1].to_json(orient="index")),
               "confusion_matrix": evaluations[name][2].tolist(),
               "description": model_notes[name], "training_minutes": minutes_by_model[name],
               "previous_accuracy": baseline_accuracy[name]}
        for name in MODEL_ORDER
    },
}
RESULTS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Machine-readable results saved: {RESULTS_PATH}")